# Demo: Transforming Time-Series Data


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
# Load and clean the bike-share data (cleaning was covered in the previous module)
df = pd.read_csv("../data/bikeshare_rides.csv", parse_dates=["date"])
df = df.groupby("date", as_index=False)["rides"].mean()
df["date"] = pd.to_datetime(df["date"])
mask = (df["date"].dt.year == 2023) & (df["date"].dt.month == 10) & (df["rides"] < 1000)
df.loc[mask, "rides"] = df.loc[mask, "rides"] * 24
df.loc[df["rides"] < 0, "rides"] = np.nan
df = df.set_index("date").asfreq("D")
df["rides"] = df["rides"].interpolate(method="time")
rides = df["rides"]
print(f"Clean series: {len(rides)} days")


## Rolling Statistics


In [ ]:
window = 30
rmean = rides.rolling(window).mean()
rstd = rides.rolling(window).std()

fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color=UN["Light Gray"])
axes[0].plot(rmean.index, rmean.values, color=UN["Black"], label="30-day mean")
axes[0].set_ylabel("Rides", fontsize=16)
axes[0].legend()
axes[0].set_title("Rolling Mean", fontsize=18, fontweight="bold")
axes[1].plot(rstd.index, rstd.values, color=UB["Medium Blue"])
axes[1].set_ylabel("Rides", fontsize=16)
axes[1].set_title("Rolling Std", fontsize=18, fontweight="bold")
plt.tight_layout()
axes[0].tick_params(axis="both", labelsize=14)
axes[1].tick_params(axis="both", labelsize=14)
plt.show()



## The ADF Test


In [ ]:
result = adfuller(rides.dropna(), autolag="AIC")
print(f"ADF statistic: {result[0]:.4f}")
print(f"p-value:       {result[1]:.6f}")
print(f"Stationary:    {'yes' if result[1] < 0.05 else 'no'}")


## Differencing


In [ ]:
diffed = rides.diff().dropna()

result_diff = adfuller(diffed, autolag="AIC")
print(f"Differenced ADF p-value: {result_diff[1]:.6f}")
print(f"Stationary: {'yes' if result_diff[1] < 0.05 else 'no'}")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
axes[0].plot(rides.index, rides.values, color=UN["Black"])
axes[0].set_ylabel("Rides", fontsize=16)
axes[0].set_title("Original", fontsize=18, fontweight="bold")
axes[1].plot(diffed.index, diffed.values, color=US["Green"])
axes[1].axhline(y=0, color=UN["Medium Gray"], linestyle="--", alpha=0.5)
axes[1].set_ylabel("Daily change", fontsize=16)
axes[1].set_title("First-differenced", fontsize=18, fontweight="bold")
plt.tight_layout()
axes[0].tick_params(axis="both", labelsize=14)
axes[1].tick_params(axis="both", labelsize=14)
plt.show()



## Train/Test Split


In [ ]:
test_days = 90
train, test = rides.iloc[:-test_days], rides.iloc[-test_days:]
print(f"Train: {len(train)} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test:  {len(test)} days ({test.index[0].date()} to {test.index[-1].date()})")
